# Results: quality per token

Answers the research questions in `SPEC.md` §1 from eval runs:

1. **RQ1** How much token cost can be cut before quality degrades?
2. **RQ2** Learned router vs. heuristic — is the extra complexity worth it?
3. **RQ3** Where does semantic caching help, and where does it silently serve wrong answers?
4. **RQ4** What does context compression cost in quality per token saved?

**Inputs.** A manifest (`[analysis] manifest`, default `results/manifest.toml`) maps roles to eval run directories:

```toml
routing = "results/<run over evals/seed.jsonl>"          # RQ1-RQ3
compression = "results/<run over evals/conversations.jsonl>"  # RQ4
```

**Method** (config `[analysis]`): paired bootstrap confidence intervals over items; a config *holds quality* if the lower CI bound of (config − reference) is above −margin. All numbers come from `gpt_efficient.analysis`; this notebook only displays them. Runs made with `--fake` are flagged **ILLUSTRATIVE** throughout. Interpretation paragraphs are left as TODO for a human to write from real runs.

In [ ]:
from pathlib import Path

from IPython.display import Image, Markdown, display

from gpt_efficient.analysis import (
    cache_analysis,
    cheapest_held,
    config_table,
    load_manifest,
    load_run,
    write_findings,
)
from gpt_efficient.config import Settings

settings = Settings()
cfg = settings.analysis
runs = {role: load_run(path) for role, path in load_manifest(cfg.manifest).items()}
for role, run in runs.items():
    flag = "FAKE (illustrative only)" if run.meta.fake else "real"
    print(f"{role:12} {flag:26} {run.path}  experiments={run.experiments}")

## Findings

The full writeup (`findings.md`) with all four research questions, generated from the runs above.

In [ ]:
paths = write_findings(runs, settings, cfg.out_dir)
display(Markdown(paths["findings"].read_text()))

## Figures

In [ ]:
for key in ("frontier_cost_ci", "frontier_tokens_ci", "compression_amortized", "compression_oneshot"):
    if key in paths:
        display(Image(filename=str(paths[key])))

## Exploration: where did the cheapest quality-holding config lose quality?

Per-item comparison against the reference, worst first — raw material for the RQ1 interpretation.

In [ ]:
routing = runs.get("routing")
if routing is not None:
    ref, table = config_table(routing, cfg)
    best = cheapest_held(table, "cost")
    target = best.experiment if best else next((r.experiment for r in table if r.experiment != ref), None)
    if ref and target:
        ref_q = {r.item_id: r for r in routing.rows(ref) if r.quality is not None}
        drops = sorted(
            ((r.quality - ref_q[r.item_id].quality, r) for r in routing.rows(target)
             if r.quality is not None and r.item_id in ref_q),
            key=lambda d: d[0],
        )
        print(f"{target} vs {ref} (worst 10):")
        for delta, r in drops[:10]:
            print(f"  {delta:+.2f}  {r.item_id:12} {r.difficulty:6} tier={r.tier.value:8} {r.rationale[:80]}")

## Exploration: wrong cache hits

Each wrong hit with the cached answer that was served — raw material for RQ3.

In [ ]:
if routing is not None:
    _, wrong = cache_analysis(routing, cfg, settings.eval.low_quality)
    by_key = {(r.experiment, r.item_id): r for r in routing.results}
    for w in wrong:
        r = by_key[(w.experiment, w.item_id)]
        print(f"{w.experiment} / {w.item_id}  sim={w.cache_sim}  judge={w.judge_score}  tags={w.tags}")
        print(f"    served: {(r.answer or '')[:160]!r}")
    if not wrong:
        print("no wrong cache hits")